In [0]:
%sql
CREATE CATALOG IF NOT EXISTS rmg
MANAGED LOCATION 'abfss://bronze@rmgdevstorage123.dfs.core.windows.net/_managed';
CREATE SCHEMA IF NOT EXISTS rmg.bronze;
CREATE SCHEMA IF NOT EXISTS rmg.silver;
CREATE SCHEMA IF NOT EXISTS rmg.gold;
CREATE SCHEMA IF NOT EXISTS rmg.quarantine;
CREATE SCHEMA IF NOT EXISTS rmg.ops;

In [0]:
STORAGE = "rmgdevstorage123"
p = f"abfss://bronze@{STORAGE}.dfs.core.windows.net/_smoketest"
spark.range(5).write.mode("overwrite").format("delta").save(p)
print("rows:", spark.read.format("delta").load(p).count()) # → 5

In [0]:
dbutils.secrets.get("rmg-kv", "postgres-password")

In [0]:
pip install psycopg2-binary sqlalchemy

In [0]:
pip install azure-identity azure-keyvault-secrets

In [0]:
dbutils.library.restartPython()

In [0]:
import psycopg2
from psycopg2 import sql
import pandas as pd

In [0]:

host = "psql-rmg-dev123.postgres.database.azure.com"
user = "rmgadmin"
database = "rmg_erp"
password = "646429emoN?"  # Paste password directly or use env var

# Connect
conn = psycopg2.connect(
    host=host,
    user=user,
    password=password,
    database=database,
    sslmode="require"
)

In [0]:
# Load data into Spark
query = "SELECT * FROM erp.suppliers"
df_pandas = pd.read_sql(query, conn)

# Convert to Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)
display(df_spark)

conn.close()

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS control;
CREATE TABLE control.ingestion_config (
config_id SERIAL PRIMARY KEY,
source_schema TEXT NOT NULL,
source_table TEXT NOT NULL,
target_path TEXT NOT NULL,
load_type TEXT NOT NULL CHECK (load_type IN ('full','incremental')),
watermark_column TEXT,
last_watermark TIMESTAMP DEFAULT '1900-01-01',
is_active BOOLEAN DEFAULT TRUE
);
CREATE TABLE control.pipeline_run_log (
run_log_id SERIAL PRIMARY KEY,
run_id TEXT,
source_table TEXT,
start_time TIMESTAMP DEFAULT now(),
end_time TIMESTAMP,
status TEXT,
rows_read BIGINT,
error_message TEXT
);
INSERT INTO control.ingestion_config
(source_schema, source_table, target_path, load_type, watermark_column) VALUES
('erp','suppliers', 'erp/suppliers', 'full', NULL),
('erp','factories', 'erp/factories', 'full', NULL),
('erp','purchase_orders', 'erp/purchase_orders', 'incremental','updated_at'),
('erp','production_orders','erp/production_orders','incremental','updated_at');
-- Without these, every "incremental" extract is a full table scan
CREATE INDEX idx_po_updated ON erp.purchase_orders (updated_at);
CREATE INDEX idx_pro_updated ON erp.production_orders (updated_at);